# 🐍 Неделя 2. Семинар продвинутого потока.

**Сегодня в программе:**
1. Абстрактные и метаклассы: зачем они нужны в реальных фреймворках.
2. Протоколы: как они упрощают поддержку кода.
3. Класс vs экземпляр: `__class__`, `type`, `isinstance`, `__dict__`.
4. Подводные камни наследования: MRO, `super()`, множественное наследование.

In [ ]:
from __future__ import annotations
import abc
from typing import Protocol, runtime_checkable, Iterable, Optional, Dict, Any
import math

## 🟢 1. Абстрактные классы и метаклассы.

### Абстрактные классы (`abc.ABC` и `abc.ABCMeta`)

* В Python **абстрактный класс** — это класс, который **нельзя создать напрямую**.
* Он задаёт **интерфейс**: набор методов, которые должны реализовать его наследники.
* Для создания абстрактных классов используют модуль `abc`:
  * `ABC` — базовый класс для абстрактных классов.
  * `ABCMeta` — метакласс, управляющий их поведением.
* Методы в абстрактном классе помечаются декоратором `@abstractmethod`.

**Пример:**

```python
import abc

class Shape(abc.ABC):
    @abc.abstractmethod
    def area(self):
        pass

# Shape()  # ошибка: нельзя создать абстрактный класс

class Circle(Shape):
    def __init__(self, r):
        self.r = r

    def area(self):
        return 3.14 * self.r**2

c = Circle(5)  # работает, так как реализован метод area
```

Таким образом, абстрактный класс — это **шаблон**, а его наследники обязаны заполнить «пустые места».

---

### Метаклассы

* В Python **всё является объектами**, даже классы.
* Метакласс — это «класс для классов», который управляет тем, **как создаются сами классы**.
* Обычный класс создаёт экземпляры, а метакласс создаёт сами классы.

По умолчанию у всех классов метакласс `type`:

```python
class A:
    pass

print(type(A))   # <class 'type'> → значит, класс A создан метаклассом type
```

Можно задать свой метакласс через `metaclass=...` и добавить туда особую логику.


### Задание 1.
**Идея:**
Давайте создадим **метакласс**, который автоматически собирает все классы-наследники в один реестр. Это удобно, чтобы потом быстро находить все доступные «плагины» или «агрегаторы». ТЗ:

1. **Создайте класс `RegistryMeta`**, наследуйте его от `abc.ABCMeta`.

   * Так мы сможем работать и с абстрактными классами.

2. **Заведите словарь `registry`** внутри метакласса.

   * Он будет хранить все зарегистрированные классы.

3. **Переопределите метод `__new__`**:

   * `__new__` вызывается при создании класса (не объекта!).
   * Внутри создаём сам класс через `super().__new__`.
   * Если класс **не помечен как абстрактный**, добавляем его в `registry`.

4. **Проверьте работу**:

   * Создадим абстрактный базовый класс с `metaclass=RegistryMeta`.
   * Создадим несколько наследников.
   * Выведем `RegistryMeta.registry` и убедимся, что они там появились.

In [ ]:
class RegistryMeta(abc.ABCMeta):
    """Метакласс, который записывает классы-агрегаторы в реестр."""
    registry: Dict[str, type] = {}

    def __new__(mcls, name, bases, namespace):
        cls = super().__new__(mcls, name, bases, namespace)
        # Регистрируем все подклассы кроме самого базового абстрактного класса
        if not namespace.get('__abstract__', False):
            RegistryMeta.registry[name] = cls
        return cls

### Задание 2.
**Идея:**
Сделайте базовый класс для всех агрегаторов, который задаёт обязательный интерфейс (`update`, `result`) и работает вместе с нашим `RegistryMeta`.

1. Создайте базовый класс `Aggregator` и укажите для него метакласс `RegistryMeta`.
2. Добавьте служебный флаг `__abstract__ = True`, чтобы этот класс **не попадал** в реестр.
3. Определите абстрактный метод `update(value)`, который будет принимать новое число.
4. Определите абстрактный метод `result()`, который возвращает итоговое значение (или `None`, если данных нет).
5. Добавьте метод `reset()`, который можно переопределять в наследниках (по умолчанию — пустой).

In [ ]:
class Aggregator(metaclass=RegistryMeta):
    __abstract__ = True  # чтобы RegistryMeta знал не регистрировать этот базовый класс

    @abc.abstractmethod
    def update(self, value: float) -> None:
        """Добавить одно значение в агрегатор."""
        raise NotImplementedError

    @abc.abstractmethod
    def result(self) -> Optional[float]:
        """Вернуть финальное значение агрегата (None если пусто)."""
        raise NotImplementedError

    def reset(self) -> None:
        """Сброс (по-умолчанию ничего не делает)."""
        pass

### Задание 3.

**Идея:** Наконец создадим конкретные классы-агрегаторы, которые умеют считать разные статистики по числовым данным и автоматически регистрируются в реестре через `RegistryMeta`.

1. Создайте класс `CountAggregator`, который:

   * хранит количество добавленных значений,
   * реализует метод `update` для увеличения счётчика,
   * реализует метод `result`, возвращающий текущее количество.

2. Создайте класс `MeanAggregator`, который:

   * хранит текущее среднее и количество значений,
   * обновляет среднее онлайн при каждом вызове `update` (по формуле Welford),
   * возвращает среднее через `result`, или `None`, если данных ещё нет.

3. Создайте класс `MinMaxAggregator`, который:

   * хранит минимум и максимум из добавленных значений,
   * обновляет минимум и максимум при каждом `update`,
   * возвращает кортеж `(min, max)` через `result`, или `None`, если данных ещё нет.

4. Проверьте, что новые агрегаторы автоматически появились в реестре `RegistryMeta.registry`.

In [ ]:
class CountAggregator(Aggregator):
    def __init__(self):
        self.n = 0

    def update(self, value: float) -> None:
        self.n += 1

    def result(self) -> int:
        return self.n


class MeanAggregator(Aggregator):
    """Онлайн-среднее по Welford'у."""
    def __init__(self):
        self.n = 0
        self.mean = 0.0

    def update(self, value: float) -> None:
        self.n += 1
        delta = value - self.mean
        self.mean += delta / self.n

    def result(self) -> Optional[float]:
        return self.mean if self.n > 0 else None


class MinMaxAggregator(Aggregator):
    def __init__(self):
        self.min = math.inf
        self.max = -math.inf
        self._empty = True

    def update(self, value: float) -> None:
        if value < self.min:
            self.min = value
        if value > self.max:
            self.max = value
        self._empty = False

    def result(self) -> Optional[tuple]:
        return None if self._empty else (self.min, self.max)

**Самопроверка!**

In [ ]:
# Тестовые данные
data = [10, 20, 30, 40, 50]

# Все агрегаторы зарегистрированы??
print("Зарегистрированные агрегаторы:", list(RegistryMeta.registry.keys()))

# Экземпляры агрегаторов
count_agg = CountAggregator()
mean_agg = MeanAggregator()
minmax_agg = MinMaxAggregator()

# Добавляем данные
for value in data:
    count_agg.update(value)
    mean_agg.update(value)
    minmax_agg.update(value)

print("CountAggregator result:", count_agg.result())
print("MeanAggregator result:", mean_agg.result())
print("MinMaxAggregator result:", minmax_agg.result())

# Проверка метода reset (по умолчанию ничего не делает, просто вызываем)
count_agg.reset()
mean_agg.reset()
minmax_agg.reset()
print("Reset вызван успешно для всех агрегаторов.")

Зарегистрированные агрегаторы: ['CountAggregator', 'MeanAggregator', 'MinMaxAggregator']
CountAggregator result: 5
MeanAggregator result: 30.0
MinMaxAggregator result: (10, 50)
Reset вызван успешно для всех агрегаторов.


## 🟢 2. Протоколы.

**Duck-typing:**
В Python тип объекта часто не имеет значения - важна только поддержка нужного интерфейса (методов и атрибутов).

*Принцип заключается в следующем:*

> *"Если объект умеет делать то, что требуется, его можно использовать."*

*Пример: функция может работать с любым объектом, который реализует метод `update`.*

---

**Протокол (`Protocol`):**

* Это формальный способ описать интерфейс объекта.
* Определяет методы и атрибуты, которые объект должен иметь.
* Не требует наследования.
* Позволяет статическую проверку типов (например, с `mypy`).

*Пример:*

```python
from typing import Protocol

class AggregatorProtocol(Protocol):
    def update(self, value: float) -> None: ...
    def result(self) -> float: ...
```

Любой объект с методами `update` и `result` удовлетворяет этому протоколу.

**Проверка в рантайме (`@runtime_checkable`):**
обычно протоколы используются только для статической проверки типов (анализ кода без запуска).
Если нужно проверить во время выполнения, что объект соответствует протоколу, используют декоратор `@runtime_checkable`.

**Зачем это нужно:**

* Гарантирует интерфейс, но не связывает с конкретным классом.
* Упрощает работу с разными реализациями агрегаторов.

### Задание 4.
Опишите протокол для объектов-агрегаторов, у которых есть методы `update` и `result`.

Сделайте его проверяемым во время выполнения с помощью `@runtime_checkable`.

In [ ]:
@runtime_checkable
class SummarizerProtocol(Protocol):
    def update(self, value: float) -> None: ...
    def result(self) -> Optional[float]: ...

## 🟢 3. Служебные методы.

В Python у любого объекта есть специальные «служебные» инструменты для работы с его типом и внутренним устройством.  
Рассмотрим пока четыре ключевых механизма: `__dict__`, `__class__`, `isinstance` и `type`.

* `__dict__` — это словарь атрибутов объекта.  
  Хранит все данные экземпляра (переменные, которые мы задаём в `__init__` и позже).  
  Позволяет «заглянуть внутрь» объекта и увидеть его текущее состояние.

* `__class__` — ссылка на класс, по которому был создан объект.  
  Помогает понять, к какому классу принадлежит данный экземпляр.

* `isinstance(obj, Class)` — функция для проверки, является ли объект экземпляром указанного класса (или его наследников).  
  Удобно использовать для безопасных проверок типов.

* `type(obj)` — возвращает точный класс объекта.  
  В отличие от `isinstance`, не учитывает наследование и всегда даёт именно тот класс, в котором объект был создан.

### Задание 5.

Ниже дан код класса `Column`, который хранит числовые значения и композиционно использует агрегаторы. Создайте экземпляр класса с набором агрегаторов и добавьте в него несколько числовых значений.
Затем продемонстрируйте внутренние инструменты Python для работы с объектами:

1. **`__dict__`** — выведите словарь атрибутов экземпляра.
2. **`__class__`** — покажите, к какому классу принадлежит объект.
3. **`isinstance`** — проверьте, является ли объект и его агрегаторы экземплярами определённых классов.
4. **`type`** — получите точный тип объекта и сравните его с классом `Column`.

In [ ]:
class Column:
    """
    Хранит числовые значения и композиционно использует агрегаторы.
    """
    def __init__(self, name: str, aggregators: Optional[Iterable[Aggregator]] = None):
        self.name = name
        self._values: list[float] = []
        # композиция: каждый Column сам держит экземпляры агрегаторов
        self.aggregators = list(aggregators) if aggregators else [
            CountAggregator(), MeanAggregator(), MinMaxAggregator()
        ]

    def add(self, value: float) -> None:
        """Добавляет одно значение, обновляя агрегаторы."""
        # Можно добавлять проверки в дочерних классах (validate), здесь — простая логика
        self._values.append(value)
        for agg in self.aggregators:
            agg.update(value)

    def extend(self, values: Iterable[float]) -> None:
        for v in values:
            self.add(v)

    def summarize(self) -> Dict[str, Any]:
        """Возвращает словарь с результатами всех агрегаторов."""
        summary = {}
        for agg in self.aggregators:
            key = agg.__class__.__name__
            summary[key] = agg.result()
        return summary

    # пару дандер-методов как у списка — удобно для демонстрации
    def __len__(self):
        return len(self._values)

    def __iter__(self):
        return iter(self._values)

    def __getitem__(self, idx):
        return self._values[idx]

    def __repr__(self):
        return f"Column(name={self.name!r}, n={len(self)})"


In [ ]:
# создаём колонку с данными
col = Column("numbers", [CountAggregator(), MeanAggregator()])
col.extend([1.0, 2.0, 3.0])

# __dict__ — внутренний словарь атрибутов экземпляра
print("__dict__:", col.__dict__)

# __class__ — ссылка на сам класс
print("__class__:", col.__class__)

# isinstance — проверка принадлежности к классу
print("isinstance(col, Column):", isinstance(col, Column))
print("isinstance(col.aggregators[0], CountAggregator):", isinstance(col.aggregators[0], CountAggregator))

# type — возвращает тип объекта
print("type(col):", type(col))
print("type(col) is Column:", type(col) is Column)

__dict__: {'name': 'numbers', '_values': [1.0, 2.0, 3.0], 'aggregators': [<__main__.CountAggregator object at 0x7f9ed221a7c0>, <__main__.MeanAggregator object at 0x7f9ed221ac40>]}
__class__: <class '__main__.Column'>
isinstance(col, Column): True
isinstance(col.aggregators[0], CountAggregator): True
type(col): <class '__main__.Column'>
type(col) is Column: True


### Задание 6.

Закрепим работу с метаклассами на логичном примере:
* Вспомним метакласс `RegistryMeta`, который автоматически регистрирует все классы-наследники в общий словарь `registry` и классы-агрегаторы, использующие этот метакласс.
* Реализуйте функцию `show_aggregator_registry()`, которая выводит все зарегистрированные агрегаторы.

In [ ]:
def show_aggregator_registry():
    print("Registered Aggregators (via RegistryMeta):")
    for name in RegistryMeta.registry:
        print("  -", name)

## 🟢 4. `super`-наследование.

>Разберём метод `super()`: в Python он позволяет обращаться к методам родительского класса из дочернего. Это удобно, когда нужно:
>* **Вызвать конструктор родителя** для корректной инициализации атрибутов.
>* **Использовать методы родителя** без дублирования кода.
>* **Расширить функциональность**: сначала выполняем логику родителя, потом добавляем свою.

>Проще говоря, `super()` помогает **не переписывать код родителя**, а дополнять или модифицировать его в наследнике.

### Задание 7.

Создайте класс `ValidatedColumn`, который наследуется от `Column` и добавляет проверку значений при добавлении.

1. Переопределите метод `add`, чтобы перед добавлением каждого значения проверять его корректность.
2. Реализуйте метод `validate`, который:

   * проверяет, что значение является числом (`int` или `float`),
   * проверяет, что значение не меньше `min_value` (если задано),
   * проверяет, что значение не больше `max_value` (если задано).
3. В методе `add` используйте `super()` для вызова базового метода `add` после успешной валидации.

In [ ]:
class ValidatedColumn(Column):
    """
    Валидация значений при добавлении (переопределение add).
    Показывает использование super() для расширения поведения.
    """
    def __init__(self, name: str, min_value: Optional[float] = None, max_value: Optional[float] = None,
                 aggregators: Optional[Iterable[Aggregator]] = None):
        super().__init__(name=name, aggregators=aggregators)
        self.min_value = min_value
        self.max_value = max_value

    def validate(self, value: float) -> None:
        if not isinstance(value, (int, float)):
            raise TypeError(f"Value must be numeric, got {type(value)}")
        if self.min_value is not None and value < self.min_value:
            raise ValueError(f"Value {value} < min_value {self.min_value}")
        if self.max_value is not None and value > self.max_value:
            raise ValueError(f"Value {value} > max_value {self.max_value}")

    def add(self, value: float) -> None:
        # расширяем поведение: валидируем, потом вызываем базовый add
        self.validate(value)
        super().add(value)

## 🟢 5. Множественное-наследование.

* В Python класс может наследоваться от нескольких базовых классов одновременно.
* Это позволяет комбинировать поведение нескольких родителей в одном классе.
* Синтаксис простой:

```python
class D(B, C):
    ...
```

где `D` наследует методы и атрибуты как от `B`, так и от `C`.

При множественном наследовании могут возникать конфликты методов с одинаковыми именами. Эти конфликты разрешаются по принципу **MRO (Method Resolution Order)** — упорядоченному поиску методов в иерархии классов:

* **MRO** определяет порядок, в котором Python ищет методы и атрибуты при наследовании.
* Для определения порядка поиска методов при множественном наследовании используется алгоритм **C3 linearization**. Он гарантирует, что MRO будет линейным, детерминированным и сохранит порядок базовых классов, указанный при объявлении класса
* Пример для класса `D(B, C)`:

```
D → B → C → A → object
```

* Это значит, что если вызвать `D().smth()`, Python выполнит:

```text
D.smth() → B.smth() → C.smth() → A.smth()
```

* Каждый вызов `super()` идет по следующему элементу в MRO.


### Задание 8.
Реализуйте несколько классов с множественным наследованием и переопределением методов, чтобы продемонстрировать принцип **MRO (Method Resolution Order)** в Python:

1. Создайте базовый класс `A` с методом `whoami`, который возвращает строку `"A"`.
2. Создайте классы `B` и `C`, наследующие `A`, и переопределите метод `whoami`, добавляя свою идентификацию и вызывая метод родителя через `super()`.
3. Создайте класс `D`, который наследует одновременно от `B` и `C`.
4. Проверьте вызов метода `whoami` у экземпляра `D` и проследите порядок вызовов методов согласно **MRO**.

In [ ]:
class A:
    def whoami(self):
        return "A"

class B(A):
    def whoami(self):
        return "B -> " + super().whoami()

class C(A):
    def whoami(self):
        return "C -> " + super().whoami()

class D(B, C):
    # MRO у D: D, B, C, A, object
    pass

## 🟢 6. Демонстрация

Вот как бы выглядел код в `main.py`-подобном файле. Инкапсуляция во всей красе:)

In [ ]:
def demo():
    print("=== Demo: Column и агрегаторы ===")
    col = Column("ages")
    col.extend([23, 35, 31, 40, 29])
    print("Колонка:", col)
    print("len, first, items:", len(col), col[0], list(col))
    print("summarize:", col.summarize())

    # показать __dict__, __class__, type, isinstance
    print("\nИнспекция:")
    print("col.__dict__:", col.__dict__)
    print("col.__class__:", col.__class__)
    print("type(col):", type(col))
    print("isinstance(col, Column):", isinstance(col, Column))

    # ValidatedColumn
    print("\nValidatedColumn (min=0, max=100). Попытка добавить 150 -> ошибка")
    vcol = ValidatedColumn("scores", min_value=0, max_value=100)
    vcol.extend([10, 20, 30])
    try:
        vcol.add(150)
    except ValueError as e:
        print("Ожидаемая ошибка:", e)
    print("vcol summary:", vcol.summarize())

    # MRO demo
    print("\nMRO demo:")
    d = D()
    print("D.whoami():", d.whoami())
    print("D MRO:", [c.__name__ for c in D.__mro__])

    # Registry demo
    print()
    show_aggregator_registry()

    # Проверки (простые)
    assert col.summarize()["CountAggregator"] == 5
    mean = col.summarize()["MeanAggregator"]
    assert abs(mean - (23 + 35 + 31 + 40 + 29) / 5) < 1e-9

    print("\nAll asserts passed. Demo finished.")


if __name__ == "__main__":
    demo()

=== Demo: Column и агрегаторы ===
Колонка: Column(name='ages', n=5)
len, first, items: 5 23 [23, 35, 31, 40, 29]
summarize: {'CountAggregator': 5, 'MeanAggregator': 31.6, 'MinMaxAggregator': (23, 40)}

Инспекция:
col.__dict__: {'name': 'ages', '_values': [23, 35, 31, 40, 29], 'aggregators': [<__main__.CountAggregator object at 0x7f9ed220b160>, <__main__.MeanAggregator object at 0x7f9ed220b9a0>, <__main__.MinMaxAggregator object at 0x7f9ed220b970>]}
col.__class__: <class '__main__.Column'>
type(col): <class '__main__.Column'>
isinstance(col, Column): True

ValidatedColumn (min=0, max=100). Попытка добавить 150 -> ошибка
Ожидаемая ошибка: Value 150 > max_value 100
vcol summary: {'CountAggregator': 3, 'MeanAggregator': 20.0, 'MinMaxAggregator': (10, 30)}

MRO demo:
D.whoami(): B -> C -> A
D MRO: ['D', 'B', 'C', 'A', 'object']

Registered Aggregators (via RegistryMeta):
  - CountAggregator
  - MeanAggregator
  - MinMaxAggregator

All asserts passed. Demo finished.


# 🌟 До скорых встреч!